# Average Shot Length Visualizations

This notebook reads the completed PySceneDetect shot-length results, joins them to `metadata-all.json`, prints film titles that do not directly match metadata titles, and builds Plotly visualizations for average shot length by film and metadata group.

In [1]:
# Optional Plotly dependency check.
# Set AUTO_INSTALL_PLOTLY=True if this notebook kernel does not already have Plotly.
AUTO_INSTALL_PLOTLY = True

import importlib
import subprocess
import sys

missing = []
for module_name, package_name in {"plotly": "plotly", "pandas": "pandas", "nbformat": "nbformat"}.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing.append(package_name)

if missing and AUTO_INSTALL_PLOTLY:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
    missing = []
    for module_name, package_name in {"plotly": "plotly", "pandas": "pandas", "nbformat": "nbformat"}.items():
        try:
            importlib.import_module(module_name)
        except ModuleNotFoundError:
            missing.append(package_name)

if missing:
    packages = " ".join(missing)
    raise ModuleNotFoundError(
        "Missing package(s): " + packages + "\n\n"
        "This Jupyter kernel is running:\n"
        f"  {sys.executable}\n\n"
        "Install into this exact kernel with a notebook cell like:\n"
        f"  import sys\n  !{{sys.executable}} -m pip install -U {packages}\n"
    )

print("Plotly/Pandas/nbformat dependencies look ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.1/354.1 kB 3.2 MB/s eta 0:00:00a 0:00:01
Plotly/Pandas/nbformat dependencies look ready.


In [2]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
import plotly.express as px

SHOT_LONG_CSV = Path("./shot_length_results/shot_lengths_long.csv")
MOVIE_PROGRESS_CSV = Path("./shot_length_results/movie_progress.csv")
METADATA_JSON = Path("./metadata-all.json")

TOP_N = 100
SELECTED_CATEGORY = "Northeast"

print(f"Shot CSV: {SHOT_LONG_CSV.resolve()}")
print(f"Progress CSV: {MOVIE_PROGRESS_CSV.resolve()}")
print(f"Metadata JSON: {METADATA_JSON.resolve()}")

Shot CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/shot_lengths_long.csv
Progress CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/movie_progress.csv
Metadata JSON: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/metadata-all.json


In [3]:
# Special cases where the video filename differs from metadata-all.json title punctuation.
# Cells that need metadata titles should call metadata_lookup_title_for_video_title().
SPECIAL_TITLE_MAPPINGS = {
    "买牛扒瓜园": "买牛、扒瓜园",
    "大李小李和老李": "大李、小李和老李",
    "打铜锣补锅": "打铜锣、补锅",
    "特快列车": "“特快”列车",
    "老保管新媳妇来了": "老保管、新媳妇来了",
}


def metadata_lookup_title_for_video_title(video_title: str) -> str:
    """Return the metadata title corresponding to a video/CSV film title."""
    video_title = str(video_title).strip()
    return SPECIAL_TITLE_MAPPINGS.get(video_title, video_title)


print("Special title mappings:")
for video_title, metadata_title in SPECIAL_TITLE_MAPPINGS.items():
    print(f"- {video_title}  ->  {metadata_title}")


Special title mappings:
- 买牛扒瓜园  ->  买牛、扒瓜园
- 大李小李和老李  ->  大李、小李和老李
- 打铜锣补锅  ->  打铜锣、补锅
- 特快列车  ->  “特快”列车
- 老保管新媳妇来了  ->  老保管、新媳妇来了


In [4]:
def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


def normalize_title(title: str) -> str:
    title = str(title).strip()
    title = re.sub(r"\s+", "", title)
    return re.sub(r"[、，,。\.\"'“”‘’《》〈〉（）()\[\]【】\-—·:：;；!！?？]", "", title)


def load_metadata_records(path: Path) -> list[dict]:
    payload = json.loads(path.read_text())
    if isinstance(payload, dict) and "films" in payload:
        return payload["films"]
    if isinstance(payload, list):
        return payload
    raise ValueError("metadata-all.json must contain a top-level 'films' list or be a list")


def production_names(record: dict) -> list[str]:
    names = []
    for item in as_list(record.get("production")):
        if isinstance(item, dict):
            name = item.get("name") or item.get("eng") or item.get("chn")
        else:
            name = item
        if name not in (None, ""):
            names.append(str(name))
    return names


def metadata_row(record: dict, film: str, match_method: str) -> dict:
    return {
        "film": film,
        "metadata_title": record.get("title"),
        "metadata_key": record.get("key"),
        "year": record.get("year"),
        "production_names": production_names(record),
        "category": [str(x) for x in as_list(record.get("category")) if x not in (None, "")],
        "filmType": record.get("filmType"),
        "colour": record.get("colour"),
        "match_method": match_method,
    }


shot_long_df = pd.read_csv(SHOT_LONG_CSV)
progress_df = pd.read_csv(MOVIE_PROGRESS_CSV)
metadata_records = load_metadata_records(METADATA_JSON)

film_stats = (
    shot_long_df.groupby("film", as_index=False)
    .agg(
        avg_shot_length=("duration_sec", "mean"),
        median_shot_length=("duration_sec", "median"),
        shot_count=("shot_index", "count"),
        total_duration_sec=("duration_sec", "sum"),
    )
)
film_stats["avg_shot_length"] = film_stats["avg_shot_length"].round(3)
film_stats["median_shot_length"] = film_stats["median_shot_length"].round(3)
film_stats["total_duration_sec"] = film_stats["total_duration_sec"].round(3)

progress_small = progress_df[["film", "status", "video_path"]].drop_duplicates("film")
film_stats = film_stats.merge(progress_small, on="film", how="left")

title_to_records = defaultdict(list)
normalized_to_records = defaultdict(list)
for record in metadata_records:
    title = str(record.get("title", "")).strip()
    if title:
        title_to_records[title].append(record)
        normalized_to_records[normalize_title(title)].append(record)

metadata_titles = set(title_to_records)
print(f"Shot-result films: {film_stats['film'].nunique()}")
print(f"Metadata records: {len(metadata_records)}")
print(f"Metadata unique titles: {len(metadata_titles)}")

Shot-result films: 454
Metadata records: 732
Metadata unique titles: 728


## Direct Title Mismatch Check

This cell prints films present in the shot/video results whose names do not directly equal any `metadata-all.json` `title`. These are usually punctuation differences.

In [5]:
direct_unmatched = sorted(
    film for film in film_stats["film"].astype(str).str.strip().unique()
    if film not in metadata_titles
)

print(f"Direct unmatched films: {len(direct_unmatched)}")
for film in direct_unmatched:
    mapped_title = metadata_lookup_title_for_video_title(film)
    normalized_matches = [r.get("title") for r in normalized_to_records.get(normalize_title(film), [])]
    hint = mapped_title if mapped_title != film else (normalized_matches[0] if len(normalized_matches) == 1 else "")
    print(f"- {film}" + (f"  ->  {hint}" if hint else ""))


Direct unmatched films: 5
- 买牛扒瓜园  ->  买牛、扒瓜园
- 大李小李和老李  ->  大李、小李和老李
- 打铜锣补锅  ->  打铜锣、补锅
- 特快列车  ->  “特快”列车
- 老保管新媳妇来了  ->  老保管、新媳妇来了


In [6]:
def choose_record(records: list[dict]) -> dict | None:
    if not records:
        return None
    return records[0]


def match_metadata_for_film(film: str) -> dict:
    film = str(film).strip()

    exact = choose_record(title_to_records.get(film, []))
    if exact is not None:
        return metadata_row(exact, film, "exact_title")

    mapped_title = metadata_lookup_title_for_video_title(film)
    if mapped_title != film:
        mapped = choose_record(title_to_records.get(mapped_title, []))
        if mapped is not None:
            return metadata_row(mapped, film, "special_title_mapping")

    normalized_records = normalized_to_records.get(normalize_title(film), [])
    if len(normalized_records) == 1:
        return metadata_row(normalized_records[0], film, "normalized_title")
    if len(normalized_records) > 1:
        return metadata_row(normalized_records[0], film, "normalized_title_ambiguous_first")

    return {
        "film": film,
        "metadata_title": pd.NA,
        "metadata_key": pd.NA,
        "year": pd.NA,
        "production_names": [],
        "category": [],
        "filmType": pd.NA,
        "colour": pd.NA,
        "match_method": "unmatched",
    }


metadata_matches = pd.DataFrame(
    [match_metadata_for_film(film) for film in film_stats["film"].astype(str)]
)
film_level = film_stats.merge(metadata_matches, on="film", how="left")
film_level["year"] = pd.to_numeric(film_level["year"], errors="coerce").astype("Int64")

final_unmatched = film_level[film_level["match_method"] == "unmatched"]["film"].tolist()
print(f"Final unmatched after special mapping/normalization: {len(final_unmatched)}")
for film in final_unmatched:
    print(f"- {film}")

display(film_level.head(10))


Final unmatched after special mapping/normalization: 0


,film,avg_shot_length,median_shot_length,shot_count,total_duration_sec,status,video_path,metadata_title,metadata_key,year,production_names,category,filmType,colour,match_method
0,51号兵站,10.228,4.161,551,5635.709,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...,51号兵站,51 hao bing zhan,1961,[Haiyan Film Studio],[Shanghai (state)],Feature,B&W,exact_title
1,一串项链,43.339,13.462,46,1993.610,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一串项链...,一串项链,yi chuan xiang lian,1966,[Xi'an Film Studio],[Xi'an],Feature,B&W,exact_title
2,一件提案,55.762,5.280,84,4684.000,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一件提案...,一件提案,yi jian ti an,1954,[Beijing Film Studio],[Beijing],Feature,B&W,exact_title
3,一场风波,835.960,235.860,6,5015.760,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一场风波...,一场风波,yi chang feng bo,1954,[Shanghai Film Studio],[Shanghai (state)],Feature,B&W,exact_title
4,一天一夜,10.957,6.621,406,4448.487,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一天一夜...,一天一夜,yi tian yi ye,1958,[Beijing Film Studio],[Beijing],Artistic Documentary,Colour,exact_title
5,一文钱,1.654,1.000,2267,3749.964,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4,一文钱,yi wen qian,1960,[Xi'an Film Studio],[Xi'an],Opera,B&W,exact_title
6,一日千里,13.041,2.000,309,4029.559,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一日千里...,一日千里,yi ri qian li,1958,[August First Film Studio],[Beijing],Feature,B&W,exact_title
7,一百个放心,7.186,2.000,610,4383.580,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一百个放...,一百个放心,yi bai ge fang xin,1966,[August First Film Studio],[Beijing],Feature,B&W,exact_title
8,一贯害人道,11.456,6.281,559,6404.088,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一贯害人...,一贯害人道,yi guan hai ren dao,1952,[Beijing Film Studio],[Beijing],Feature,B&W,exact_title
9,一颗红心,25.900,11.882,211,5464.949,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/一颗红心...,一颗红心,yi ke hong xin,1965,[Changchun Film Studio],[Northeast],Opera,B&W,exact_title


In [7]:
def explode_list_column(df: pd.DataFrame, list_column: str, output_column: str) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        values = row[list_column]
        if not isinstance(values, list) or not values:
            values = ["Unknown"]
        for value in values:
            new_row = row.to_dict()
            new_row[output_column] = value if value not in (None, "") else "Unknown"
            rows.append(new_row)
    return pd.DataFrame(rows)


def summarize_group(df: pd.DataFrame, group_column: str) -> pd.DataFrame:
    summary = (
        df.groupby(group_column, dropna=False)
        .agg(
            avg_shot_length=("avg_shot_length", "mean"),
            median_film_avg=("avg_shot_length", "median"),
            film_count=("film", "nunique"),
            total_shots=("shot_count", "sum"),
        )
        .reset_index()
    )
    summary[group_column] = summary[group_column].fillna("Unknown")
    summary["avg_shot_length"] = summary["avg_shot_length"].round(3)
    summary["median_film_avg"] = summary["median_film_avg"].round(3)
    return summary.sort_values("avg_shot_length", ascending=False)


def show_bar(summary_df: pd.DataFrame, x: str, y: str, title: str, orientation: str = "v", height: int = 650):
    fig = px.bar(
        summary_df,
        x=x,
        y=y,
        orientation=orientation,
        title=title,
        hover_data=[col for col in ["film_count", "total_shots", "median_film_avg"] if col in summary_df.columns],
    )
    fig.update_layout(height=height, xaxis_title=x, yaxis_title=y)
    fig.show()
    return fig

## 1. Top-N Films by Average Shot Length

High values are useful for quickly finding likely detection failures or unusually slow-cut films.

In [8]:
top_films = (
    film_level.sort_values("avg_shot_length", ascending=False)
    .head(TOP_N)
    .sort_values("avg_shot_length", ascending=True)
)

fig_top = px.bar(
    top_films,
    x="avg_shot_length",
    y="film",
    orientation="h",
    title=f"Top {TOP_N} Films by Average Shot Length",
    hover_data=["shot_count", "median_shot_length", "total_duration_sec", "year", "filmType", "colour", "match_method"],
)
fig_top.update_layout(height=max(700, TOP_N * 18), xaxis_title="Average shot length (sec)", yaxis_title="Film")
fig_top.show()

## 2. Average Shot Length by Production Studio

In [9]:
production_expanded = explode_list_column(film_level, "production_names", "production_name")
production_summary = summarize_group(production_expanded, "production_name")
display(production_summary)
fig_production = show_bar(
    production_summary.sort_values("avg_shot_length", ascending=True),
    x="avg_shot_length",
    y="production_name",
    title="Average Shot Length by Production Studio",
    orientation="h",
    height=max(600, len(production_summary) * 26),
)

,production_name,avg_shot_length,median_film_avg,film_count,total_shots
13,Shanghai Film Studio,163.170,18.002,52,15956
4,Changjiang Film Studio,145.380,145.380,1,41
1,Beijing Film Studio,48.992,12.244,76,37877
0,August First Film Studio,43.291,16.160,63,23258
9,Jiangnan Film Studio,28.884,21.259,14,4108
6,Haiyan Film Studio,27.670,13.502,38,18061
16,Xi'an Film Studio,27.577,25.042,17,7708
3,Changchun Film Studio,24.602,14.557,109,45123
14,Tianma Film Studio,23.906,14.130,36,15292
15,Wenhua Film Company,21.765,12.016,3,1216


## 3. Average Shot Length by Production Region / Category

In [10]:
category_expanded = explode_list_column(film_level, "category", "category_name")
category_summary = summarize_group(category_expanded, "category_name")
display(category_summary)
fig_category = show_bar(
    category_summary.sort_values("avg_shot_length", ascending=True),
    x="avg_shot_length",
    y="category_name",
    title="Average Shot Length by Production Region / Category",
    orientation="h",
    height=max(500, len(category_summary) * 32),
)

,category_name,avg_shot_length,median_film_avg,film_count,total_shots
5,Shanghai (state),77.152,15.634,140,53417
0,Beijing,46.332,13.041,139,61062
4,Shanghai (private),31.477,11.796,8,4095
6,Xi'an,27.577,25.042,17,7708
3,Northeast,22.954,14.004,135,59823
2,Hong Kong,17.707,18.016,4,1374
1,Canton,13.506,10.641,15,7488


## 4. Average Shot Length by Film Type

In [11]:
film_type_df = film_level.copy()
film_type_df["filmType"] = film_type_df["filmType"].fillna("Unknown")
film_type_summary = summarize_group(film_type_df, "filmType")
display(film_type_summary)
fig_film_type = show_bar(
    film_type_summary,
    x="filmType",
    y="avg_shot_length",
    title="Average Shot Length by Film Type",
    height=550,
)

,filmType,avg_shot_length,median_film_avg,film_count,total_shots
3,Opera,71.088,22.625,63,22713
1,Feature,44.829,13.776,362,159847
4,Performance,25.224,14.028,11,3111
2,Musical,16.628,9.980,9,5438
0,Artistic Documentary,14.402,11.859,9,2484


## 5. Average Shot Length by Colour

In [12]:
colour_df = film_level.copy()
colour_df["colour"] = colour_df["colour"].fillna("Unknown")
colour_summary = summarize_group(colour_df, "colour")
display(colour_summary)
fig_colour = show_bar(
    colour_summary,
    x="colour",
    y="avg_shot_length",
    title="Average Shot Length by Colour",
    height=550,
)

,colour,avg_shot_length,median_film_avg,film_count,total_shots
0,B&W,55.951,14.676,346,147619
1,Colour,17.634,12.498,108,45974


## 6. Average Shot Length Trend by Year

In [13]:
year_summary = (
    film_level.dropna(subset=["year"])
    .groupby("year", as_index=False)
    .agg(
        avg_shot_length=("avg_shot_length", "mean"),
        median_film_avg=("avg_shot_length", "median"),
        film_count=("film", "nunique"),
        total_shots=("shot_count", "sum"),
    )
    .sort_values("year")
)
year_summary["avg_shot_length"] = year_summary["avg_shot_length"].round(3)
year_summary["median_film_avg"] = year_summary["median_film_avg"].round(3)
display(year_summary)

fig_year = px.line(
    year_summary,
    x="year",
    y="avg_shot_length",
    markers=True,
    title="Average Shot Length Trend by Year",
    hover_data=["film_count", "total_shots", "median_film_avg"],
)
fig_year.update_layout(height=600, xaxis_title="Year", yaxis_title="Average shot length (sec)")
fig_year.show()

,year,avg_shot_length,median_film_avg,film_count,total_shots
0,1949,15.831,10.103,10,6292
1,1950,12.086,9.350,17,8704
2,1951,31.716,15.222,11,5418
3,1952,10.053,9.497,5,3399
4,1953,25.299,23.069,6,2747
5,1954,313.922,17.887,21,8783
6,1955,38.050,15.746,22,9936
7,1956,47.697,16.160,39,13183
8,1957,26.470,15.486,36,12570
9,1958,24.041,14.630,53,17530


## 7. Average Shot Length Trend by Selected Category

In [14]:
selected_category_df = category_expanded[
    category_expanded["category_name"].astype(str).str.contains(SELECTED_CATEGORY, case=False, na=False, regex=False)
].copy()

if selected_category_df.empty:
    print(f"No films found for SELECTED_CATEGORY={SELECTED_CATEGORY!r}.")
    print("Available categories:")
    for value in sorted(category_expanded["category_name"].dropna().astype(str).unique()):
        print(f"- {value}")
else:
    selected_category_year = (
        selected_category_df.dropna(subset=["year"])
        .groupby("year", as_index=False)
        .agg(
            avg_shot_length=("avg_shot_length", "mean"),
            median_film_avg=("avg_shot_length", "median"),
            film_count=("film", "nunique"),
            total_shots=("shot_count", "sum"),
        )
        .sort_values("year")
    )
    selected_category_year["avg_shot_length"] = selected_category_year["avg_shot_length"].round(3)
    selected_category_year["median_film_avg"] = selected_category_year["median_film_avg"].round(3)
    display(selected_category_year)

    fig_selected_category = px.line(
        selected_category_year,
        x="year",
        y="avg_shot_length",
        markers=True,
        title=f"Average Shot Length Trend for Category: {SELECTED_CATEGORY}",
        hover_data=["film_count", "total_shots", "median_film_avg"],
    )
    fig_selected_category.update_layout(height=600, xaxis_title="Year", yaxis_title="Average shot length (sec)")
    fig_selected_category.show()

,year,avg_shot_length,median_film_avg,film_count,total_shots
0,1949,15.972,17.168,4,2759
1,1950,11.869,9.136,9,4944
2,1951,11.021,11.021,1,146
3,1952,11.658,11.658,2,986
4,1953,23.028,15.117,3,933
5,1954,20.447,13.775,7,4932
6,1955,12.567,8.592,8,5853
7,1956,19.108,16.100,14,6417
8,1957,15.565,11.996,12,4973
9,1958,17.104,14.630,21,6923
